# session

> Read, write, and build native Claude Code session transcripts

In [ ]:
#| default_exp session

## Session layout

Claude Code stores each conversation as JSONL at `~/.claude/projects/<dir>/<session-id>.jsonl`. `<dir>` is the resolved project path with every non-alphanumeric character replaced by `-`, including underscores. On macOS, a project in `/tmp/foo` therefore appears under `-private-tmp-foo`. `sess_dir` and `sess_file` compute these paths. A transcript is named by the conversation's first session id. `claude --resume` and post-compaction restarts can advertise a fresh id in `CLAUDE_CODE_SESSION_ID` while appending records to the original file, so the environment variable may name an id with no file behind it. `cur_sess` finds the project's most recently modified transcript.

Each line is one JSON object. The main fields are `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. Conversation records are `user` and `assistant`; a tool result is a `user` record containing `tool_result` blocks. The active conversation is the `parentUuid` chain walked back from the last record by `sess_thread`. `load_sess` and `load_recs` read every record, and `rec_txt` extracts each record's readable text.

## Building sessions

Resume does not care who wrote the file: a transcript assembled by hand resumes like any other. `mk_rec` fills one record's envelope, `save_sess` and `append_sess` chain and write records, and `msgs2recs`/`msgs2sess` convert whole Anthropic-style message lists into resumable sessions, deterministically when given a key. `mk_tu`, `mk_tr`, and `tool_turn` build synthetic tool exchanges, and `prefix_tools` qualifies caller tool names the way Claude Code records them.

Reading functions do not modify transcripts. `save_sess` and `append_sess` do: read their docs and inspect the target records before calling them; `save_sess` replaces a whole session file. `llmsurgery` builds on this module for finding, searching, curating, converting, and compacting sessions.

Claude Code stores every conversation as a JSONL transcript, and `claude --resume <session-id>` rebuilds a conversation from one. It does not care who wrote the file. A transcript assembled by hand, including tool calls that never really ran, resumes like any other. So sessions can be mined for data, saved as templates, or built synthetically to give a fresh session worked examples of tool use already in its context. This module finds, reads, and writes them.

In [ ]:
#| export
import json, os, re, uuid
from datetime import datetime, timezone
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import delegates

In [ ]:
from fastcore.test import *
from collections import Counter
import tempfile, shutil
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage, fork_session

## Where sessions live

Each project gets a folder under `~/.claude/projects`, named by the project's absolute path with every character that is not a letter or digit replaced by `-`. The path is resolved first, which matters on macOS, where `/tmp` and `/var` are symlinks into `/private`. The folder for a project in `/tmp/foo` is therefore `-private-tmp-foo`.

In [ ]:
#| export
SESSIONS = Path.home()/'.claude'/'projects'

def sess_dir(
    cwd=None, # Project directory; the current directory if None
):
    "The folder where Claude Code keeps session transcripts for the project at `cwd`"
    return SESSIONS/re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd or '.').expanduser().resolve()))

In [ ]:
sess_dir()

Path('/Users/jhoward/.claude/projects/-Users-jhoward-aai-ws-llmsurgery-nbs')

Underscores are replaced too, which is easy to get wrong when sanitizing by hand:

In [ ]:
test_eq(sess_dir('/a/b_c').name, '-a-b-c')
test_eq(sess_dir('~'), sess_dir(Path.home()))

Claude Code exports a session id as `CLAUDE_CODE_SESSION_ID` to every process it spawns, shell commands and MCP servers alike. But it identifies the current run, not the conversation: `claude --resume` and post-compaction restarts put a fresh id in the environment, while records keep appending to the original transcript, which is named by the conversation's first session id. So the env var names the transcript only while a conversation is on its first run, and can advertise an id that names no file at all. The reliable name for "this conversation" is instead the most recently modified transcript in the project's session folder: appends keep the live transcript's mtime freshest through restarts and resumes alike. `cur_sess` uses that heuristic, keeping the env var only as a fallback for when no transcript exists yet. One caveat: two conversations open on the same project trade the newest spot on every write, so under concurrent sessions pass an explicit `sid` (clikernel sidesteps this by resolving its host conversation once, at worker spawn, when the spawning conversation's transcript is freshest).

In [ ]:
#| export
def cur_sess(
    cwd=None, # Project directory; the current directory if None
):
    "The current conversation's session id: the most recent transcript for the project at `cwd`, else the advertised id"
    try: return max(sess_dir(cwd).glob('*.jsonl'), key=lambda p: p.stat().st_mtime).stem
    except ValueError: return os.environ.get('CLAUDE_CODE_SESSION_ID')

In [ ]:
cur_sess()

The newest transcript wins; with no transcripts at all, the advertised id is used:

In [ ]:
tp = Path(tempfile.mkdtemp())
sd = sess_dir(tp)
sd.mkdir(parents=True)
for i,n in enumerate(['older','newer']):
    (sd/f'{n}.jsonl').touch()
    os.utime(sd/f'{n}.jsonl', (i,i))
test_eq(cur_sess(tp), 'newer')
test_eq(cur_sess(tempfile.mkdtemp()), os.environ.get('CLAUDE_CODE_SESSION_ID'))

Prefix resolution needs a glob to yield exactly one path. `uniq_path` returns a glob's single result, None when nothing matched, and raises when several did.

In [ ]:
#| export
def uniq_path(
    paths, # Candidate paths, e.g. from a glob
    ref, # The reference they were matched against, for the error message
):
    "The single path in `paths`, or None if there are none; raises if `ref` is ambiguous"
    res = sorted({str(o) for o in paths})
    if len(res)>1: raise ValueError(f'{ref!r} matches {len(res)} sessions:\n' + '\n'.join(res))
    return Path(res[0]) if res else None

In [ ]:
#| export
def sess_file(
    sid=None, # Session id or unique id prefix; `cur_sess(cwd)` if None
    cwd=None, # Project directory; the current directory, then all projects, if None
):
    "Path to the transcript of session `sid` for the project at `cwd`"
    sid = sid or cur_sess(cwd)
    p = sess_dir(cwd)/f'{sid}.jsonl'
    if p.exists(): return p
    root,pat = (sess_dir(cwd),f'{sid}*.jsonl') if cwd is not None else (SESSIONS,f'*/{sid}*.jsonl')
    return uniq_path(root.glob(pat), sid) or p

Under Claude Code, `sess_file()` with no arguments is therefore the running conversation's own transcript, surviving resumes and restarts. Session ids are unique across projects, so when an explicit `sid`'s file is not in the current directory's folder, `sess_file` looks across all project folders, and the defaults work from anywhere, including a notebook kernel whose working directory is not the project root.

In [ ]:
test_eq(sess_file('abc', '/a/b_c'), SESSIONS/'-a-b-c'/'abc.jsonl')

A `sid` that names no transcript is treated as an id prefix, so the eight characters that identify a session at a glance are enough to open it. An exact id still resolves without globbing, and a prefix matching more than one transcript raises rather than choosing: silently opening the wrong session is the failure worth spending an error on.

In [ ]:
test_eq(sess_file('old', tp), sd/'older.jsonl')
for n in ['dup-a','dup-b']: (sd/f'{n}.jsonl').touch()
with expect_fail(contains='matches 2 sessions'): sess_file('dup', tp)
sess_file('new', tp)

In [ ]:
#| export
def load_recs(path):
    "Session records read directly from JSONL `path`"
    return dict2obj(Path(path).read_jsonl())

## Creating dummy data

The examples below use real Claude Code transcripts rather than approximating its record format. Each fixture builder returns immediately when its target exists, so a normal notebook run is deterministic; delete the generated file or directory before rerunning when the fixture needs refreshing.

In [ ]:
#| export
ant_data = Path(files('fastclaude')/'data'/'ant')

In [ ]:
def mk_ant_fixture_project(path=None):  # chkstyle: ignore-node
    "Create the minimal Claude project used by the ant fixtures"
    assert ant_data.is_dir(), f'ant fixtures ship as package data; missing {ant_data}'
    root = Path(path) if path else ant_data/'project'
    skill = root/'.claude/skills/ant-fixture/SKILL.md'
    if skill.exists(): return root
    skill.parent.mkdir(parents=True, exist_ok=True)
    skill.write_text("""---
name: ant-fixture
description: Report the fixed ant fixture fact when asked.
---

When invoked, report exactly: `The ant fixture skill is loaded.`
""")
    return root

The fixture project supplies the smallest useful project environment: one local skill with a fixed response. The fixtures live inside the package (`fastclaude/data/ant`) and ship with it, anchored via `importlib.resources` so any working directory finds them. `ant_data` is exported, so downstream packages (llmsurgery, notably) build their own narratives on the same captures.

In [ ]:
mk_ant_fixture_project()

Path('data/ant/project')

The source fixture is a real Agent SDK session with a deliberately narrow surface. Claude loads the one project skill, runs one Bash command, evaluates one expression through the persistent clikernel MCP server, and finishes with fixed text. This captures genuine skill injection, tool calls, tool results, and Claude Code bookkeeping without unrelated session noise.

In [ ]:
async def query_sid(msgs):
    "Session id from an Agent SDK message stream"
    sid = None
    async for m in msgs:
        if isinstance(m, ResultMessage): sid = m.session_id
    if not sid: raise RuntimeError('Claude did not return a session id')
    return sid

In [ ]:
async def mk_ant_source(path=None):  # chkstyle: ignore-node
    "Create the real Claude session used by the ant fixtures"
    path = Path(path) if path else ant_data/'source.jsonl'
    if path.exists(): return path
    root = mk_ant_fixture_project(path.parent/'project')
    cmd = shutil.which('clikernel-mcp')
    if not cmd: raise FileNotFoundError('clikernel-mcp')
    opts = ClaudeAgentOptions(cwd=root.resolve(), model='haiku', system_prompt='Follow the requested tool sequence exactly.',
        tools=['Skill','Bash','mcp__clikernel__execute'], allowed_tools=['Bash','mcp__clikernel__execute'],
        skills=['ant-fixture'], setting_sources=['project'], strict_mcp_config=True,
        mcp_servers=dict(clikernel=dict(type='stdio', command=cmd)), max_turns=8)
    prompt = "Use the ant-fixture skill. Then use Bash to run `printf 'bash fixture\\n'`. Then use clikernel to evaluate `6*7`. After all tools finish, reply exactly: fixture complete."
    sid = await query_sid(query(prompt=prompt, options=opts))
    src = sess_file(sid, root)
    if not src.exists(): raise FileNotFoundError(src)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(src.read_bytes())
    return path

On the first run, `mk_ant_source` performs the model call and copies Claude Code’s transcript into the packaged `data/ant/source.jsonl`. Later runs reuse that checked-in transcript, while deleting the file explicitly requests a fresh capture against the installed Claude Code and Agent SDK versions.

In [ ]:
await mk_ant_source()

Path('data/ant/source.jsonl')

A real transcript contains both conversation and Claude Code bookkeeping. Counting record types first shows what the file actually contains before we decide which subset each reader needs.

In [ ]:
source_path = ant_data/'source.jsonl'
source_recs = load_recs(source_path)
len(source_recs),Counter(source_recs.attrgot('type'))

(20,
 Counter({'assistant': 8,
          'user': 5,
          'attachment': 3,
          'queue-operation': 2,
          'ai-title': 1,
          'last-prompt': 1}))

The user and assistant records carry the conversation. Attachments inject context such as skill contents; queue operations, titles, and prompt markers are bookkeeping retained in the JSONL but not ordinary chat messages.

### The parent chain

In [ ]:
#| export
def sess_thread(
    recs, # Session records, e.g. from `load_sess`
):
    "The records on the active conversation chain, walking `parentUuid` back from the last record"
    recs = L(r for r in recs if r.get('type') in ('user','assistant','attachment','system') and r.get('uuid'))
    byid = {r.uuid:r for r in recs}
    cur,res,seen = recs[-1],[],set()
    while cur is not None:
        if cur.uuid in seen: raise ValueError(f'parentUuid cycle at record uuid {cur.uuid}')
        seen.add(cur.uuid)
        res.append(cur)
        cur = byid.get(cur.get('parentUuid'))
    return L(reversed(res))

Claude reconstructs a conversation by starting at the final linked conversation record and walking `parentUuid` backwards. Some bookkeeping records, including `custom-title`, carry UUIDs but are not part of that chain, so `sess_thread` considers only user, assistant, attachment, and system records.

In [ ]:
source_thread = sess_thread(source_recs)
len(source_recs),len(source_thread),Counter(source_thread.attrgot('type'))

(20, 16, Counter({'assistant': 8, 'user': 5, 'attachment': 3}))

The four omitted records are unlinked bookkeeping. The active chain retains conversation records and injected-context attachments, and every consecutive pair must link correctly.

In [ ]:
assert all(b.parentUuid==a.uuid for a,b in zip(source_thread,source_thread[1:]))

Breaking a parent link makes everything before it unreachable, which mirrors what resume does with abandoned or malformed branches. A *cyclic* chain (possible only in a corrupt file, e.g. one holding duplicate uuids) raises instead of walking forever - and the writers refuse to create such a file in the first place:

In [ ]:
broken = L(obj2dict(r) for r in source_thread)
broken[-2]['parentUuid'] = None
test_eq(len(sess_thread(L(dict2obj(r) for r in broken))), 2)
cyc = L(dict2obj(obj2dict(r)) for r in source_thread)
cyc[-1]['parentUuid'] = cyc[-1]['uuid']
test_fail(lambda: sess_thread(cyc), contains='cycle')

In [ ]:
#| export
def _txts(o, skip=()):
    if isinstance(o, str): yield o
    elif isinstance(o, dict): yield from (t for k,v in o.items() if k not in skip for t in _txts(v, skip))
    elif is_listy(o): yield from (t for x in o for t in _txts(x, skip))

In [ ]:
#| export
def rec_txt(
    r, # A session record
):
    "Every readable string in `r`'s message content, joined, for finding records by text"
    return '\n'.join(_txts(obj2dict(r).get('message', {}).get('content', ''), skip=('type','id','tool_use_id','signature')))

### Forking

In [ ]:
#| export
def sess_id(recs):
    "The session id in transcript records `recs`"
    return first(r.get('sessionId') for r in recs if r.get('sessionId'))

The fork fixture is derived from the checked-in source transcript using the Agent SDK’s native `fork_session`. The source is first installed into the fixture project’s Claude session directory, because the SDK operates on normal Claude Code storage rather than arbitrary JSONL paths.

In [ ]:
#| eval: false
from claude_agent_sdk import fork_session

In [ ]:
async def mk_ant_fork(path=None, source=None):  # chkstyle: ignore-node
    "Create the native Claude fork used by the ant fixtures"
    path,source = Path(path) if path else ant_data/'fork.jsonl',Path(source) if source else ant_data/'source.jsonl'
    if path.exists(): return path
    await mk_ant_source(source)
    root = mk_ant_fixture_project(source.parent/'project')
    recs = load_recs(source)
    sid = sess_id(recs)
    live = sess_file(sid, root)
    live.parent.mkdir(parents=True, exist_ok=True)
    live.write_bytes(source.read_bytes())
    forked = fork_session(sid, directory=str(root.resolve()), title='ant fixture fork')
    src = sess_file(forked.session_id, root)
    if not src.exists(): raise FileNotFoundError(src)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(src.read_bytes())
    return path

`mk_ant_fork` stores the native fork beside it as `fork.jsonl`. Later runs reuse that fixture; deleting it requests a fresh fork from the installed SDK.

The stored fixture path confirms that the native fork is available for the examples below.

In [ ]:
#| eval: false
await mk_ant_fork()

Path('data/ant/fork.jsonl')

Loading both transcripts shows the new session identity and the top-level records produced by the fork.

In [ ]:
fork_path = ant_data/'fork.jsonl'
fork_recs = load_recs(fork_path)
source_sid,fork_sid = sess_id(source_recs),sess_id(fork_recs)
source_sid,fork_sid,len(source_recs),len(fork_recs),Counter(fork_recs.attrgot('type'))

('e12b4bd3-4f36-4dda-873d-2ff25d1f1044',
 '5d2b9564-f7f8-48c3-b611-f33fbe39c577',
 20,
 17,
 Counter({'assistant': 8, 'user': 5, 'attachment': 3, 'custom-title': 1}))

The fork has a fresh session ID. Its transcript contains the copied session records and the requested `custom-title`; unlinked bookkeeping from the captured source is absent.

In [ ]:
source_thread,fork_thread = sess_thread(source_recs),sess_thread(fork_recs)
len(source_thread),len(fork_thread),Counter(source_thread.attrgot('type')),Counter(fork_thread.attrgot('type'))

(16,
 16,
 Counter({'assistant': 8, 'user': 5, 'attachment': 3}),
 Counter({'assistant': 8, 'user': 5, 'attachment': 3}))

The active chains have the same record-type structure. Native forking assigns fresh record UUIDs and rebuilds the parent links under the fork’s session ID.

In [ ]:
test_ne(source_sid, fork_sid)
test_eq(source_thread.attrgot('type'), fork_thread.attrgot('type'))
test_eq(set(fork_thread.attrgot('sessionId')), {fork_sid})
test_ne(source_thread.attrgot('uuid'), fork_thread.attrgot('uuid'))
test_eq(fork_thread.attrgot('parentUuid'), [None,*fork_thread.attrgot('uuid')[:-1]])
[(a.uuid,b.uuid) for a,b in zip(source_thread[:3],fork_thread[:3])]

[('f711b566-2404-41a5-a1b6-36f99d52e391',
  '58718dc0-80ed-4298-befc-c933e7913a94'),
 ('071ddef8-7dfb-42dc-b32c-38506751d50e',
  '1e84ec59-9412-44c1-8f12-16aeef9a201c'),
 ('05dd6db4-a8a1-422a-9617-ea7b19a00873',
  '2773c527-c1a1-4816-9a3f-435a9cde9f45')]

Record envelopes change, while the readable conversation remains the same. Rendering the fork gives a direct view of the session that later compaction examples will use.

In [ ]:
test_eq([rec_txt(r) for r in source_thread], [rec_txt(r) for r in fork_thread])

## Writing a session

Records are plain dicts, so writing a session comes down to filling the envelope and linking the chain. `mk_rec` fills the envelope for one message. It writes the optional bookkeeping a real transcript carries (`version`, `gitBranch`, `userType`, `entrypoint`, `session_id`, API metadata on assistant records, and `toolUseResult` mirroring a tool result's content, which is what the transcript UI renders), not only the six required fields: what Claude Code's LLM side makes of a sparse-but-valid record is close to untestable, so we err towards realistic.

Two records with the same content get different files by default, since ids and timestamps are fresh each call. Sometimes the opposite is wanted: the same history should produce byte-identical records, so the same session id maps to the same file however many times it is rebuilt. `canon` gives a canonical JSON rendering to hash, and `stable_uuid` turns any string into a deterministic uuid. `fastllm_claude_code.core` derives its session and record ids this way, and a session template built from a fixed script can too.

In [ ]:
#| export
CC_VERSION = '2.1.223'

def canon(o):
    "Canonical compact JSON for `o`, key-sorted, for stable hashing"
    return json.dumps(o, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def stable_uuid(s):
    "A uuid deterministically derived from string `s`"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, s))

Canonical JSON ignores dictionary insertion order, while `stable_uuid` maps the same key to the same UUID and different keys to different UUIDs.

In [ ]:
test_eq(canon(dict(b=1, a=2)), canon(dict(a=2, b=1)))
test_eq(stable_uuid('x'), stable_uuid('x'))
assert stable_uuid('x') != stable_uuid('y')

In [ ]:
#| export
def _now(): return datetime.now(timezone.utc).strftime(r'%Y-%m-%dT%H:%M:%S.%f')[:-3]+'Z'

def _est_toks(o):
    "Rule-of-thumb token estimate for every string in `o`: words * 1.5"
    return int(sum(len(s.split()) for s in _txts(o))*1.5)

With ids, timestamps, and token estimates in hand, `mk_rec` fills one message's complete record:

In [ ]:
#| export
def mk_rec(
    role, # 'user' or 'assistant'
    content, # A string, or a list of content blocks
    cwd='.', # Project directory recorded in the envelope
    uid=None, # Record uuid; random if None
    ts=None, # ISO8601 timestamp; the current time if None
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    input_toks=0, # `input_tokens` recorded in assistant usage, e.g. an estimate of the context so far
    **kwargs, # Extra or overriding envelope fields, e.g. `isCompactSummary=True`
):
    "A transcript record for one conversation message, ready for `save_sess`"
    uid = uid or str(uuid.uuid4())
    if isinstance(content, list):
        content = [{k:v for k,v in b.items() if k!='cache_control'} if isinstance(b, dict) else b for b in content]
        if role=='user' and content and all(isinstance(b, dict) and b.get('type')=='text' for b in content): content = ''.join(b['text'] for b in content)
    msg = dict(type='message', role=role, content=content)
    r = dict(type=role, uuid=uid, parentUuid=None, sessionId=None, timestamp=ts or _now(), cwd=str(Path(cwd).expanduser().resolve()),
        version=CC_VERSION, gitBranch='HEAD', isSidechain=False, userType='external', entrypoint='cli', session_id=None, message=msg)
    if role=='user' and isinstance(content, list) and (trs := [b for b in content if isinstance(b, dict) and b.get('type')=='tool_result']):
        c = trs[-1]['content']
        r['toolUseResult'] = c if isinstance(c, list) else [dict(type='text', text=c)]
    if role=='assistant':
        tu = isinstance(content, list) and any(isinstance(b, dict) and b.get('type')=='tool_use' for b in content)
        r['requestId'] = 'req_'+stable_uuid(f'{uid}:req').replace('-', '')[:24]
        usage = dict(input_tokens=input_toks, output_tokens=_est_toks(content), cache_creation_input_tokens=0, cache_read_input_tokens=0)
        msg.update(id='msg_'+stable_uuid(f'{uid}:msg').replace('-', '')[:24],
            stop_reason='tool_use' if tu else 'end_turn', stop_sequence=None, stop_details=None, usage=usage)
        if model: msg['model'] = model
    return dict(r, **kwargs)

In [ ]:
mk_rec('user', 'Hello!')

{'type': 'user',
 'uuid': '37c5febb-1af0-4afc-b44a-9df9eaed0fa4',
 'parentUuid': None,
 'sessionId': None,
 'timestamp': '2026-07-17T05:27:38.969Z',
 'cwd': '/Users/jhoward/aai-ws/llmsurgery/nbs',
 'version': '2.1.206',
 'gitBranch': 'HEAD',
 'isSidechain': False,
 'userType': 'external',
 'permissionMode': 'default',
 'message': {'type': 'message', 'role': 'user', 'content': 'Hello!'}}

Assistant records get deterministic API metadata derived from the record id, and `stop_reason` reflects a trailing tool call:

In [ ]:
tu = [dict(type='tool_use', id='toolu_01', name='probe', input={})]
r = mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z')
test_eq(r['message']['stop_reason'], 'tool_use')
test_eq(r, mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z'))
assert 'requestId' in r and 'requestId' not in mk_rec('user', 'hi')
test_eq(r['message']['usage']['output_tokens'], _est_toks(tu))
assert 'model' not in mk_rec('assistant', tu, model=None)['message']
test_eq(mk_rec('user', 'hi', cwd='~')['cwd'], str(Path.home()))

API-side messages carry artifacts a real transcript never contains: `cache_control` markers, and text-only user turns split into blocks where Claude Code records one plain string. `mk_rec` scrubs both, so a written session is indistinguishable from one Claude Code recorded itself.

In [ ]:
cb = [dict(type='text', text='Hi ', cache_control=dict(type='ephemeral')), dict(type='text', text='there.')]
test_eq(mk_rec('user', cb)['message']['content'], 'Hi there.')
test_eq(mk_rec('user', [dict(type='text', text='hi')])['message']['content'], 'hi')
tr = mk_rec('user', [dict(type='tool_result', tool_use_id='t1', content='ok', cache_control=dict(type='ephemeral'))])
test_eq(tr['message']['content'], [dict(type='tool_result', tool_use_id='t1', content='ok')])

`save_sess` assigns a session id, chains each record to the one before, and writes the file where `claude --resume` will look for it. It re-links `parentUuid` unconditionally, so it is for writing linear conversations. Records without a `uuid` (Claude Code bookkeeping such as `last-prompt` or `mode`) ride along unchained, so a loaded real transcript can be edited and written back. To copy a session while keeping its branch structure, write the records yourself.

In [ ]:
#| export
def _chain(recs, sid, prev, ts, used=()):
    "Chain `recs` for session `sid` after uuid `prev`, refusing duplicate uuids (a cycle on disk otherwise)"
    seen = set(used)
    for r in recs:
        if 'uuid' not in r: continue   # bookkeeping records (last-prompt, mode, ...) ride along unchained
        if (u := r['uuid']) in seen: raise ValueError(f'duplicate record uuid {u}')
        seen.add(u)
        r['sessionId'],r['parentUuid'],prev = sid,prev,u
        if ts: r['timestamp'] = _now() if ts is True else ts
        if 'session_id' in r: r['session_id'] = sid

def save_sess(
    recs, # Records in conversation order, e.g. from `mk_rec`
    sid=None, # Session id; a fresh uuid if None
    cwd=None, # Project directory; the current directory if None
    ts=None, # If given, stamp every record's timestamp: True for the current time, or an ISO8601 string
):
    "Chain `recs`, write them as session `sid` for the project at `cwd`, and return `sid`"
    sid = sid or str(uuid.uuid4())
    _chain(recs, sid, None, ts)
    f = sess_file(sid, cwd or '.')
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(''.join(xdumps(r)+'\n' for r in recs))
    return sid

In [ ]:
#| export
def append_sess(
    recs, # Records to append, e.g. a munged template round
    sid=None, # Session to append to; `cur_sess()` if None
    cwd=None, # Project directory; the current directory if None
    ts=None, # If given, stamp each appended record's timestamp: True for the current time, or an ISO8601 string
):
    "Chain `recs` onto the tail of session `sid` and append them to its transcript, returning `sid`"
    sid = sid or cur_sess(cwd)
    old = [r['uuid'] for r in load_sess(sid, cwd) if 'uuid' in r]
    _chain(recs, sid, last(old), ts, old)
    with sess_file(sid, cwd).open('a') as f: f.writelines(xdumps(r)+'\n' for r in recs)
    return sid

`append_sess` is the mid-life counterpart: it chains new records onto the transcript's tail without rewriting the existing bytes, so structures already in the file (compaction records, sidechains) are left untouched. It's how a worked round gets spliced into an existing conversation, e.g. llmdojo's `claudedojo -r` refreshing a session after a compaction.

Whole conversations convert in one call: `msgs2recs` takes Anthropic-style messages (dicts with `role` and `content`, like `claude_mk_msg` or fastllm's `denorm_msgs` produce) and builds one deterministic record per message, ready for `save_sess`. The same messages and key give the same ids, so a rebuilt session file is byte-identical.

In [ ]:
#| export
def msgs2recs(
    msgs, # Anthropic-style messages: dicts with `role` and `content`
    key='', # Salt: the same messages and key give the same ids
    cwd='.', # Project directory recorded in the envelopes
    ts='2026-01-01T00:00:00.000Z', # Timestamp for every record
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    **kwargs, # Extra envelope fields for every record, e.g. `entrypoint`
):
    "Deterministic transcript records for `msgs`, one record per message"
    out,tot = [],0
    for i,m in enumerate(msgs):
        out.append(mk_rec(m['role'], m['content'], cwd=cwd, uid=stable_uuid(f'{key}:{i}:{canon(m)}'), ts=ts, model=model, input_toks=tot, **kwargs))
        tot += _est_toks(m['content'])
    return out

Converting the same messages with the same key is deterministic. Changing the key changes record ids, while roles and accumulated token estimates follow the message sequence.

In [ ]:
den = [dict(role='user', content='Ping?'), dict(role='assistant', content=[dict(type='text', text='Pong.')])]
r1,r2 = msgs2recs(den, 'k'),msgs2recs(den, 'k')
test_eq(canon(r1), canon(r2))
test_ne(r1[0]['uuid'], msgs2recs(den, 'other')[0]['uuid'])
test_eq([r['type'] for r in r1], ['user','assistant'])
test_eq(r1[1]['message']['usage'], dict(input_tokens=1, output_tokens=3, cache_creation_input_tokens=0, cache_read_input_tokens=0))

`msgs2recs` and `save_sess` compose into the one-call operation a stateless caller wants: file this message list as a session, and return the id to resume. The sid is derived from the messages and a salt, so the same history maps to the same file—replaying a conversation is idempotent rather than littering the project with near-duplicate transcripts.

In [ ]:
#| export
@delegates(msgs2recs)
def msgs2sess(
    msgs, # Anthropic-style messages: dicts with `role` and `content`
    key='', # Salt: the same messages and key give the same session id
    cwd='.', # Project directory the session is filed under
    extra=None, # Records appended after the messages, e.g. from `mk_deferred`
    **kwargs
):
    "Write `msgs` as a resumable session with a content-stable id, returning the sid"
    sid = stable_uuid(f'{key}:{canon(msgs)}:{canon(extra or [])}')
    return save_sess(msgs2recs(msgs, key=sid, cwd=cwd, **kwargs)+list(extra or []), sid, cwd)

In [ ]:
tp2 = Path(tempfile.mkdtemp())
psid = msgs2sess(den, key='pingpong', cwd=tp2)
test_eq(msgs2sess(den, key='pingpong', cwd=tp2), psid)
back = load_recs(sess_file(psid, tp2))
test_eq(back[-1].message.content[0].text, 'Pong.')
test_eq(sess_thread(back).attrgot('uuid'), back.attrgot('uuid'))

## Synthetic tool calls

A worked tool call is two records joined by one id: a `tool_use` block in an assistant record, answered by a `tool_result` block in the user record that follows. `mk_tu` and `mk_tr` build the pair so the ids cannot drift, and `tool_turn` assembles the full exchange, from request to reply.

In [ ]:
#| export
def mk_tu(
    name, # Tool name, as the transcript records it
    input=None, # Tool arguments
    tid=None, # tool_use id; random if None
):
    "A `tool_use` content block"
    return dict(type='tool_use', id=tid or 'toolu_'+uuid.uuid4().hex[:24], name=name, input=input or {})

def mk_tr(
    tu, # The `tool_use` block being answered
    content, # The tool's output
):
    "The `tool_result` content block answering `tu`"
    return dict(type='tool_result', tool_use_id=tu['id'], content=content)

def tool_turn(
    prompt, # The user request
    name, # Tool name
    input, # Tool arguments
    output, # Tool result
    answer, # The assistant's closing text
    **kwargs, # Passed to each `mk_rec`, e.g. `cwd`
):
    "A complete synthetic tool-use turn, as four records ready for `save_sess`"
    tu = mk_tu(name, input)
    return [mk_rec('user', prompt, **kwargs), mk_rec('assistant', [tu], **kwargs),
        mk_rec('user', [mk_tr(tu, output)], **kwargs), mk_rec('assistant', [dict(type='text', text=answer)], **kwargs)]

The sample session in the next section is built from exactly one such turn.

## A sample session

The smallest useful synthetic history is a tool call that never ran, whose result carries a fact the model could not know any other way. We write it against a scratch project directory.

In [ ]:
proj = Path(tempfile.mkdtemp())
sample = tool_turn('Measure the flux please.', 'flux_meter', {}, 'flux: 41.7 kilofinches',
    'The flux reading is 41.7 kilofinches.', cwd=proj)
sid = save_sess(sample, cwd=proj)
sid

'6885e8fa-4b16-458d-9087-80eeff30e397'

A real transcript interleaves bookkeeping records that carry no `uuid` (`last-prompt`, `mode`, `file-history-snapshot`, ...). `save_sess` chains around them: they keep their place in the file, untouched, so a loaded session survives an edit-and-write-back.

In [ ]:
bk = dict(type='last-prompt', prompt='Measure the flux please.')
bsid = save_sess([sample[0], bk, *sample[1:]], cwd=proj)
back = load_recs(sess_file(bsid, proj))
test_eq(back[1], bk)
test_eq(back[2].parentUuid, back[0].uuid)
bsid

## Reading a session

A transcript is one JSON object per line. `load_sess` wraps each in `dict2obj` so fields read as attributes.

In [ ]:
#| export
def load_sess(
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory if None
):
    "The records of session `sid`, as an `L` of attribute-access dicts"
    return load_recs(sess_file(sid, cwd))

Reading it back gives exactly what we wrote:

In [ ]:
back = load_sess(sid, proj)
test_eq(len(back), 4)
test_eq(back[-1].message.content[0].text, 'The flux reading is 41.7 kilofinches.')
test_eq(back[2].message.content[0].tool_use_id, back[1].message.content[0].id)

A record carries more than resume strictly needs. Only six fields are required: `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. The rest is optional bookkeeping. Strip `timestamp` and the session is not even found. `message` is shaped exactly as the Anthropic API shapes messages: a `role`, plus `content` as either a string or a list of content blocks (`text`, `tool_use`, `tool_result`, `thinking`). Assistant records in real transcripts also carry API metadata (`requestId`, `message.id`, `model`, usage), and none of it is needed on resume. In particular, synthetic histories work without `thinking` blocks.

The proof that a written session works is still a resume. The sample below resumes the deliberately synthetic flux session because that section tests session writing; it spends tokens, so it is excluded from automated notebook runs.

In [ ]:
#| eval: false
opts = ClaudeAgentOptions(resume=sid, cwd=str(proj), model='haiku')
async for m in query(prompt='What is the flux reading? Reply with only the value.', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

41.7 kilofinches


## Qualified tool names

On the wire, tools hosted by a caller are named `mcp__<server>__<name>`, and a resumed history containing earlier calls must use those same qualified names or Claude won't connect them to the tools now on offer. `prefix_tools` qualifies past `tool_use` blocks, leaving alone names that are already qualified and Claude Code's own tools (e.g. `WebSearch`), which run under their bare names.

In [ ]:
#| export
def prefix_tools(
    msgs, # Anthropic-style messages
    prefix, # Stub name prefix, e.g. 'mcp__probe__'
    skip=(), # Native tool names to leave unqualified, e.g. 'WebSearch'
):
    "Copy of `msgs` with client `tool_use` names qualified by `prefix`"
    def _fix(b):
        if not (isinstance(b, dict) and b.get('type')=='tool_use'): return b
        nm = b.get('name','')
        if not nm or nm.startswith('mcp__') or nm in skip: return b
        return dict(b, name=prefix+nm)
    return [dict(m, content=[_fix(b) for b in m['content']]) if isinstance(m.get('content'), list) else m for m in msgs]

In [ ]:
tmsgs = [dict(role='assistant', content=[dict(type='text', text='Checking.'), dict(type='tool_use', id='t1', name='flux_meter', input={})]),
    dict(role='assistant', content=[dict(type='tool_use', id='t2', name='WebSearch', input=dict(query='flux'))])]
pref = prefix_tools(tmsgs, 'mcp__probe__', skip=['WebSearch'])
test_eq([b['name'] for m in pref for b in m['content'] if b['type']=='tool_use'], ['mcp__probe__flux_meter','WebSearch'])
test_eq(tmsgs[0]['content'][1]['name'], 'flux_meter')

## Cleanup

Remove the sample from `~/.claude/projects`, along with the scratch project.

In [ ]:
shutil.rmtree(sess_dir(proj))
shutil.rmtree(proj)

In [ ]:
#| hide
#| eval: false
import nbdev
nbdev.nbdev_export()